In [ ]:
# ============================================================================
# PARAMETERS — this cell is identical in all three notebooks.
# ============================================================================
NOTEBOOK_NAME = "03_evaluate_and_build_assets"    # identity + build fingerprint of THESE cells;
NOTEBOOK_BUILD = "67fa6409ed03"  # checked against the repo so stale cells fail loudly
RUN_MODE = "micro"          # "smoke" | "micro" (default) | "budget" | "full"
NUM_GPUS = None             # None = use every visible GPU; set 1 to force single-GPU
PUBLISH_KAGGLE_DATASET = True
CKPT_DATASET_SLUG = "dentex-repro-ckpts"
DATA_DATASET_SLUG = "dentex-repro-data"
REPO_URL = "https://github.com/christopherh-88/HierarchicalDet.git"

import os, subprocess, sys

# On Kaggle the repo is cloned into /kaggle/working (the only writable place
# that survives "Save Version"); locally the notebook already sits inside it.
if os.path.isdir("/kaggle/working"):
    CLONE = "/kaggle/working/repo"
    if os.path.isdir(os.path.join(CLONE, ".git")):
        subprocess.run(["git", "-C", CLONE, "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "50", REPO_URL, CLONE], check=True)
    PROJECT_ROOT = os.path.join(CLONE, "dentex-repro")
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.environ["RUN_MODE"] = RUN_MODE
print("project root:", PROJECT_ROOT)


In [ ]:
# ---- Environment: install, pin, and prove the VENDORED code is what loaded ----
# Kaggle reverts to its base image every session, so this runs every time.
import json
from src import setup_env

# `git pull` above refreshed src/ and configs_repro/ -- but NOT these cells,
# which are the copy uploaded to Kaggle. Fail loudly rather than run a mix.
print("notebook build:", setup_env.assert_notebook_current(NOTEBOOK_NAME, NOTEBOOK_BUILD))
setup_env.install_dependencies()
# The vendored pycocotools ships Python sources only; its compiled `_mask`
# extension is grafted in here and VERIFIED BY IMPORT. It is compiled against
# numpy's C ABI, so a mismatch surfaces as "numpy.dtype size changed" deep
# inside detectron2.structures — which reads as a detectron2 problem and is not.
import numpy
print("numpy {} | pycocotools _mask -> {}".format(
    numpy.__version__, setup_env.ensure_pycocotools_mask()))
run = setup_env.bootstrap(RUN_MODE)

from src import manifest, train_utils

NUM_GPUS = NUM_GPUS if NUM_GPUS is not None else max(1, train_utils.visible_gpus())
lock = setup_env.write_requirements_lock()
environment = setup_env.env_report()
manifest.record_environment(environment)

# The repo vendors MODIFIED detectron2 / pycocotools (multi-label partial
# annotations, 3-tier category schema). A pip-installed copy silently shadows
# them and every number changes, so this is an assertion, not a warning.
found = setup_env.assert_vendored()
for module, path in found.items():
    print("{:14s} -> {}".format(module, path))
import detectron2, pycocotools, evaluator                              # noqa: F401
from hierarchialdet.util.coco_3class_eval import COCOEvaluator         # noqa: F401
from hierarchialdet.dataset_mapper_patched import DiffusionDetDatasetMapper  # noqa: F401
print("full import chain OK | commit {} | {} GPU process(es)".format(
    environment["repo_commit"][:12], NUM_GPUS))


In [ ]:
# ---- What can this session do, and what was trained? ----
from src import data_convert, degradations, eval_utils, figures, registration, tables

HAS_GPU = setup_env.gpu_report().get("available", False)
print("GPU available:", HAS_GPU, "->", "full pass" if HAS_GPU
      else "CPU pass (benchmark + asset build only)")

paths = data_convert.layout()
CFG = {name: os.path.join(setup_env.CONFIGS_REPRO, "diffdet.dentex.{}.yaml".format(name))
       for name in ("quadrant", "enumeration", "diagnosis", "base_diffusiondet")}
# The final training run (notebook 04) supersedes notebook 02's summary when
# both exist; either satisfies the same contract, and everything below also
# falls back to resolving runs by name so a summary is never load-bearing.
training = (setup_env.read_notebook_summary("04_final_training_run")
            or setup_env.read_notebook_summary("02_train_all") or {})
weights = training.get("weights") or {}
FULL_WEIGHTS = train_utils.locate_weights("diagnosis_full")

TIER_CLASSES = {0: 4, 1: 8, 2: 4}
models = {}
# Notebook 02 recorded absolute paths under /kaggle/working/runs/... . This
# session starts with that directory EMPTY and the checkpoints mounted
# read-only under /kaggle/input/, so resolve by run name instead of trusting
# the recorded path -- otherwise every model silently drops out and the
# evaluation runs on nothing.
for label, run_name, config in (("Stage_quadrant", "quadrant_stage", CFG["quadrant"]),
                                ("Stage_enumeration", "enumeration_stage", CFG["enumeration"])):
    path = train_utils.locate_weights(run_name)
    if path:
        models[label] = {"weights": path, "config": config, "tier": 2,
                         "overrides": [], "json_override": None, "run_name": run_name}
for variant in (weights.get("variants") or setup_env.VARIANT_SWITCHES):
    run_name = "diagnosis_{}".format(variant)
    path = train_utils.locate_weights(run_name)
    if path:
        models[setup_env.VARIANT_LABELS[variant]] = {
            "weights": path, "config": CFG["diagnosis"], "tier": 2,
            "overrides": [], "json_override": None, "run_name": run_name}
# Probed, not read from the summary: a base model trained by a session whose
# summary was lost would otherwise silently drop out of the evaluation.
for tier in (0, 1, 2):
    run_name = "base_diffusiondet_tier{}".format(tier)
    path = train_utils.locate_weights(run_name)
    if path:
        models["DiffusionDet_base_tier{}".format(tier)] = {
            "weights": path, "config": CFG["base_diffusiondet"], "tier": 0,
            "overrides": ["MODEL.DiffusionDet.NUM_CLASSES",
                          "[{}, 8, 4]".format(TIER_CLASSES[tier])],
            "json_override": data_convert.flat_json_path(tier, "test"),
            "flat_tier": tier, "run_name": run_name}
for label, spec in models.items():
    where = "attached" if setup_env.KAGGLE_INPUT in spec["weights"] else "local"
    print("{:26s} [{}] {}".format(label, where, spec["weights"]))
if HAS_GPU:
    assert models, "no trained checkpoints found — run notebook 02, or attach its dataset"


In [ ]:
# ---- Main evaluation: every model, every tier, three inference seeds ----
# One inference pass at tier 2 scores all three tiers, because the authors'
# inference_on_dataset stores the predictions once and evaluates tiers 0..k.
if HAS_GPU:
    for label, spec in models.items():
        print("\n=== {} ({} seeds) ===".format(label, len(run.eval_seeds)))
        payload = eval_utils.evaluate_multi_seed(
            spec["weights"], spec["config"], run.eval_seeds,
            split="diagnosis_test", tier=spec["tier"], overrides=spec["overrides"],
            limit=run.eval_limit, json_override=spec["json_override"],
            image_dir_override=paths["img_test"] if spec["json_override"] else None)
        payload["label"] = label
        payload["flat_tier"] = spec.get("flat_tier")
        eval_utils.save_result(manifest.result_name("eval_main", model=label), payload)
        for tier, cells in payload["aggregate"].items():
            ap = cells.get("AP", {})
            # Print the detection count alongside AP: AP=0 with thousands of
            # detections means the model predicts but never matches ground
            # truth, while AP=0 with zero detections is a different failure
            # entirely, and the number is the only way to tell them apart.
            detections = (payload["runs"][0]["tiers"].get(tier) or {}).get("num_predictions")
            if ap.get("mean") is not None:
                print("  {:12s} AP {:6.2f} ± {:.2f}   AR {:>6}   detections {}".format(
                    tier, ap["mean"], ap["std"],
                    "{:.3f}".format(cells["AR"]["mean"])
                    if cells["AR"]["mean"] is not None else "n/a", detections))
            else:
                print("  {:12s} AP n/a (no finite value)      detections {}".format(
                    tier, detections))


In [ ]:
# ---- Runtime, zero-detection and degenerate-box accounting ----
if HAS_GPU:
    for label, spec in models.items():
        output = os.path.join(setup_env.RUNS_DIR, "predictions", "{}_test.json".format(label))
        summary_i = eval_utils.dump_predictions(
            spec["weights"], spec["config"], "diagnosis_test", spec["tier"], output,
            seed=run.eval_seeds[0], limit=run.eval_limit, overrides=spec["overrides"],
            json_override=spec["json_override"],
            image_dir_override=paths["img_test"] if spec["json_override"] else None)
        eval_utils.save_result(
            manifest.result_name("runtime", model=label, device=summary_i["device"],
                                 step=summary_i["sample_step"]), summary_i)
        print("{:26s} {:.3f} s/image  {} images with no detections  failures {}".format(
            label, summary_i["mean_seconds_per_image"] or float("nan"),
            summary_i["images_with_no_detections"], summary_i["failure_counts"]))


In [ ]:
# ---- Checkpoint trajectory: is the ablation ordering stable over training? ----
# Each variant at ~1/3, ~2/3 and 1/1 of its budget. Stable ordering across
# checkpoints is evidence the ablation conclusion is not an artifact of the
# shortened schedule; unstable ordering is itself the finding.
if HAS_GPU:
    # Prefer notebook 02's summary, but fall back to the run records on disk:
    # a checkpoint dataset built from runs/ alone has every weight and every
    # record but no summary, and the sweep would otherwise silently do nothing.
    records = [r for r in training.get("records", []) if r.get("kind")]
    if not records:
        records = train_utils.discover_runs()
        print("summary absent; rebuilt {} run record(s) from disk".format(len(records)))
    for record in records:
        if record.get("kind") != "diagnosis_variant":
            continue
        variant = record["variant"]
        max_iter = record.get("max_iter") or record.get("plan", {}).get("max_iter")
        points = []
        for iteration, name in sorted((record.get("trajectory") or {}).items(),
                                      key=lambda kv: int(kv[0])):
            checkpoint = train_utils.locate_weights(record["name"], name + ".pth")
            if not checkpoint:
                print("missing trajectory checkpoint (skipped): {}/{}".format(
                    record["name"], name))
                continue
            payload = eval_utils.evaluate_checkpoint(
                checkpoint, CFG["diagnosis"], split="diagnosis_test", tier=2,
                seed=run.eval_seeds[0], limit=run.eval_limit)
            points.append({"iteration": int(iteration), "checkpoint": name,
                           "progress": round(int(iteration) / max_iter, 3),
                           "result": payload})
            print("{:20s} @{:>6} iters  diagnosis AP {}".format(
                variant, iteration, payload["tiers"]["diagnosis"]["metrics"]["AP"]))
        final = eval_utils.load_result(
            manifest.result_name("eval_main", model=setup_env.VARIANT_LABELS[variant]))
        if final:
            points.append({"iteration": max_iter, "checkpoint": "model_final",
                           "progress": 1.0, "result": final["runs"][0],
                           "aggregate": final["aggregate"]})
        eval_utils.save_result(manifest.result_name("trajectory", variant=variant),
                               {"variant": variant, "max_iter": max_iter, "points": points})


In [ ]:
# ---- Robustness 1: diffusion sampling steps (accuracy AND measured latency) ----
if HAS_GPU and FULL_WEIGHTS:
    # Multi-step sampling runs the full N-step denoising loop and predicts from
    # the final step. Cross-timestep ENSEMBLING is off: the released
    # implementation of it was never finished for the multi-label fork and
    # raises UnboundLocalError on the first call (see detector.py __init__).
    if len([s for s in run.step_sweep if s > 1]):
        setup_env.log_deviation(
            "diffusion step sweep runs without cross-timestep ensembling",
            "the released ensemble path is unrunnable in the multi-label fork: "
            "inference() returns single-label variable names that do not exist "
            "here, and ddim_sample then treats the aggregate as both a tensor "
            "and a per-tier list. N-step denoising itself is unaffected",
            "03_evaluate_and_build_assets",
            impact="AP-vs-steps measures denoising depth alone, without the "
                   "multi-timestep detection ensemble the upstream single-label "
                   "model would apply")
    for step in run.step_sweep:
        payload = eval_utils.evaluate_multi_seed(
            FULL_WEIGHTS, CFG["diagnosis"], run.robustness_seeds,
            split="diagnosis_test", tier=2, sample_step=step, limit=run.eval_limit)
        eval_utils.save_result(manifest.result_name("steps", step=step), payload)
        latency = eval_utils.dump_predictions(
            FULL_WEIGHTS, CFG["diagnosis"], "diagnosis_test", 2,
            os.path.join(setup_env.RUNS_DIR, "predictions", "full_step{}.json".format(step)),
            seed=run.robustness_seeds[0], sample_step=step, limit=run.eval_limit)
        eval_utils.save_result(manifest.result_name("steps_latency", step=step), latency)
        print("steps={:2d}  diagnosis AP {:.2f}  {:.3f} s/image".format(
            step, payload["aggregate"]["diagnosis"]["AP"]["mean"],
            latency["mean_seconds_per_image"]))


In [ ]:
# ---- Robustness 2: degradation grid ----
# Downscaled images are resampled back to the original resolution, so the
# ground-truth boxes stay valid and this measures lost detail, not a changed
# coordinate system.
if HAS_GPU and FULL_WEIGHTS:
    DEGRADED_ROOT = os.path.join(setup_env.RUNS_DIR, "degraded")
    for condition in degradations.degradation_grid(run):
        image_dir = paths["img_test"] if condition["kind"] == "none" else \
            degradations.build_degraded_images(paths["img_test"], condition["kind"],
                                               condition["severity"], DEGRADED_ROOT)
        payload = eval_utils.evaluate_multi_seed(
            FULL_WEIGHTS, CFG["diagnosis"], run.robustness_seeds,
            split="diagnosis_test", tier=2, limit=run.eval_limit,
            image_dir_override=image_dir)
        payload["condition"] = condition
        eval_utils.save_result(
            manifest.result_name("degradation", condition=condition["label"]), payload)
        print("{:20s} diagnosis AP {:.2f}".format(
            condition["label"], payload["aggregate"]["diagnosis"]["AP"]["mean"]))


In [ ]:
# ---- Robustness 3: clean vs stress subsets, for every trained model ----
if HAS_GPU:
    for which in ("clean", "stress"):
        subset_json = os.path.join(paths["subsets"], "test_{}.json".format(which))
        assert os.path.exists(subset_json), "run notebook 01 first: {}".format(subset_json)
        for label, spec in models.items():
            if spec["json_override"]:
                continue          # flat baselines have their own ground truth
            payload = eval_utils.evaluate_multi_seed(
                spec["weights"], spec["config"], run.robustness_seeds,
                split="diagnosis_test", tier=2, json_override=subset_json,
                limit=run.eval_limit)
            payload.update({"subset": which, "label": label})
            eval_utils.save_result(
                manifest.result_name("subset", model=label, subset=which), payload)
            print("{:24s} {:6s} diagnosis AP {:.2f}".format(
                label, which, payload["aggregate"]["diagnosis"]["AP"]["mean"]))


In [ ]:
# ---- Robustness 4: hierarchy fault injection ----
# The prior-tier boxes the diagnosis model consumes are deliberately corrupted.
# Inference-time injection is OFF for every other experiment (that reproduces the
# released behaviour, which never injects at inference), so the jitter=0/drop=0
# condition — not the main table — is this experiment's reference point.
if HAS_GPU and FULL_WEIGHTS:
    # Same trap as the checkpoints: notebook 02 recorded an absolute
    # /kaggle/working path that does not resolve in this session, while the file
    # itself sits in the attached dataset. Resolve by location, not by memory.
    prior_test = (training.get("noisy_boxes") or {}).get("prior_over_test")
    if not (prior_test and os.path.exists(prior_test)):
        prior_test = train_utils.locate_run_artifact(
            "noisy_boxes/enumeration_over_diagnosis_test.json")
    assert prior_test, (
        "the enumeration model's predictions over the test split were not found "
        "locally or in any attached dataset "
        "(runs/<mode>/noisy_boxes/enumeration_over_diagnosis_test.json). "
        "Notebook 02 produces it; attach its output.")
    print("prior-tier boxes:", prior_test)
    # Check before spending a GPU on it. If nothing clears the score threshold
    # the detector injects nothing and every condition below would return the
    # unperturbed model's numbers, identically -- which is what a previous run
    # of this experiment published: six conditions, one set of values.
    coverage = eval_utils.prior_box_coverage(prior_test)
    print("prior boxes usable: {boxes_kept}/{boxes_total} over {images_with_boxes} "
          "images (score >= {score_threshold})".format(**coverage))
    fault_injection_threshold = coverage["score_threshold"]
    if not coverage["boxes_kept"]:
        # 0.50 is the intended threshold, not a fact about this checkpoint. Try
        # the highest threshold the model's own scores actually support before
        # concluding the experiment is a no-op -- and say so loudly either way.
        adaptive = eval_utils.adaptive_score_threshold(prior_test)
        coverage = eval_utils.prior_box_coverage(prior_test, adaptive["threshold"])
        fault_injection_threshold = adaptive["threshold"]
        print("0.50 was empty ({}); retried at the adaptive threshold: "
              "{boxes_kept}/{boxes_total} boxes over {images_with_boxes} images "
              "(score >= {score_threshold})".format(adaptive["reason"], **coverage))
        if coverage["boxes_kept"]:
            setup_env.log_deviation(
                "fault-injection prior-tier threshold lowered from 0.50 to {:.4f}"
                .format(fault_injection_threshold),
                "no prior-tier box scored above 0.50 at this checkpoint's training "
                "budget ({}), so the experiment as specified would inject nothing "
                "for every condition; lowered to the highest threshold this "
                "checkpoint's own score distribution supports, chosen by "
                "eval_utils.adaptive_score_threshold, not hand-picked".format(
                    adaptive["reason"]),
                "03_evaluate_and_build_assets",
                impact="fault-injection AP numbers below are conditioned on this "
                       "lower threshold, not the paper's implied 0.50; the perturbed "
                       "boxes are lower-confidence than intended")
if HAS_GPU and FULL_WEIGHTS and coverage["boxes_kept"]:
    for condition in degradations.fault_grid(run):
        with eval_utils.noisy_box_inference(prior_test, jitter=condition["jitter"],
                                            drop=condition["drop"],
                                            score_threshold=fault_injection_threshold) as injection:
            payload = eval_utils.evaluate_multi_seed(
                FULL_WEIGHTS, CFG["diagnosis"], run.robustness_seeds,
                split="diagnosis_test", tier=2, limit=run.eval_limit)
        payload["condition"] = condition
        # Stored so the result proves the perturbation was actually applied.
        payload["injection"] = injection
        eval_utils.save_result(
            manifest.result_name("fault", condition=condition["label"]), payload)
        print("{:16s} ({:s}) diagnosis AP {:.2f}".format(
            condition["label"], condition["axis"],
            payload["aggregate"]["diagnosis"]["AP"]["mean"]))
elif HAS_GPU and FULL_WEIGHTS:
    print("SKIPPED: no prior-tier box scored above the threshold, so injection "
          "would be a no-op and every condition would return the clean baseline.")
if HAS_GPU and FULL_WEIGHTS:
    setup_env.log_deviation(
        "inference-time prior-tier box injection used for the fault-injection experiment",
        "the released code injects prior-tier boxes only during training; the "
        "inference-time path exists in detector.py but is entirely commented out and "
        "referenced unpublished paths. It is off for every other experiment, so no "
        "main-table number depends on it",
        "03_evaluate_and_build_assets")


In [ ]:
# ---- Error analysis and the qualitative figures (need the images themselves) ----
if HAS_GPU and FULL_WEIGHTS:
    predictions_path = os.path.join(setup_env.RUNS_DIR, "predictions", "Ours_full_test.json")
    if not os.path.exists(predictions_path):
        eval_utils.dump_predictions(FULL_WEIGHTS, CFG["diagnosis"], "diagnosis_test", 2,
                                    predictions_path, seed=0, limit=run.eval_limit)
    # 0.50 is the intended threshold, not a fact about this checkpoint: at this
    # training budget every detection can score under it, which makes the
    # error rate undefined rather than zero. Try 0.50 first regardless.
    error_threshold_info = eval_utils.adaptive_score_threshold(predictions_path)
    if not error_threshold_info["used_ceiling"]:
        setup_env.log_deviation(
            "error-analysis / qualitative-figure score threshold lowered from "
            "0.50 to {:.4f}".format(error_threshold_info["threshold"]),
            "{}; lowered to the highest threshold this checkpoint's own score "
            "distribution supports, chosen by eval_utils.adaptive_score_threshold, "
            "not hand-picked".format(error_threshold_info["reason"]),
            "03_evaluate_and_build_assets",
            impact="the error-cluster rate and any drawn predictions below this "
                   "point are conditioned on {:.4f}, not the paper's implied 0.50 "
                   "-- these are this checkpoint's highest-confidence detections, "
                   "not necessarily confident ones".format(error_threshold_info["threshold"]))
    print("score threshold for error analysis / qualitative figures: {}"
          .format(error_threshold_info))
    # Localisation and classification stay separate: a box that overlaps a real
    # tooth but carries the wrong diagnosis is a matched detection with a class
    # disagreement, not a miss plus a false positive.
    analysis = eval_utils.error_analysis(predictions_path, paths["test_diagnosis"], tier=2,
                                         score_threshold=error_threshold_info["threshold"])
    eval_utils.save_result(manifest.result_name("errors", model="Ours_full"), analysis)
    print(json.dumps(analysis["totals"], indent=2))
    print("\nerror rate by tier (wrong label among correctly localised boxes):")
    for tier, rate in analysis["error_rate_by_tier"].items():
        boxes = analysis["labelled_boxes_by_tier"][tier]
        print("  tier {}: {} (over {} localised boxes)".format(
            tier, "n/a" if rate is None else "{:.3f}".format(rate), boxes))
    for tier, pairs in analysis["confusion"].items():
        print("  tier {} top confusions: {}".format(tier, list(pairs.items())[:5]))
# The error_clusters figure itself is drawn in the asset build, from the saved
# result rather than from this session's predictions, so it rebuilds offline.


In [ ]:
# ---- Failure gallery and curated overlays ----
if HAS_GPU and FULL_WEIGHTS:
    with open(paths["test_diagnosis"]) as handle:
        truth = json.load(handle)
    gt_by_image = {}
    for annotation in truth["annotations"]:
        gt_by_image.setdefault(annotation["image_id"], []).append(annotation)
    with open(predictions_path) as handle:
        predictions = json.load(handle)
    pred_by_image = {}
    for record in predictions:
        if record.get("score", 0) >= error_threshold_info["threshold"]:
            pred_by_image.setdefault(record["image_id"], []).append(record)
    names = {level: {c["id"]: str(c["name"])
                     for c in truth["categories_{}".format(level + 1)]} for level in range(3)}
    by_id = {image["id"]: image for image in truth["images"]}

    def panel(image_id, title):
        gt = [(a["bbox"], names[2].get(a.get("category_id_3"), ""))
              for a in gt_by_image.get(image_id, [])]
        pred = [(r["bbox"], "{}({:.2f})".format(
            names[2].get(r.get("category_id_3"), r.get("category_id_3")), r.get("score", 0)))
            for r in pred_by_image.get(image_id, [])]
        return {"image_path": os.path.join(paths["img_test"], by_id[image_id]["file_name"]),
                "title": title, "gt": gt, "pred": pred}

    # Both figures below are titled "ground truth vs prediction". With no
    # prediction above threshold they draw ground truth only, and a reader takes
    # the ground-truth boxes for the model's output landing on every tooth --
    # the figure then asserts the opposite of the result. Refuse to draw rather
    # than publish that.
    drawable_predictions = sum(len(v) for v in pred_by_image.values())
    print("predictions above threshold: {} over {} images".format(
        drawable_predictions, len(pred_by_image)))
    gallery = eval_utils.select_gallery_cases(analysis, per_category=2)
    gallery_ids, panels = {}, []
    for bucket, rows in gallery.items():
        gallery_ids[bucket] = [r["image_id"] for r in rows]
        for row in rows:
            panels.append(panel(row["image_id"], "{}\n{}".format(bucket, row["file_name"])))
    _threshold_used = error_threshold_info["threshold"]
    _threshold_caption_note = (
        "" if error_threshold_info["used_ceiling"] else
        " Score threshold: {:.4f} (lowered from 0.50 -- {}).".format(
            _threshold_used, error_threshold_info["reason"]))
    if not drawable_predictions:
        figures.record_not_run(
            "figure:qualitative", "03_evaluate_and_build_assets", run.mode,
            "no detection scored above {:.4f} ({}), so both qualitative figures "
            "would show ground-truth boxes only while captioned as "
            "prediction-vs-truth comparisons. {} ground-truth boxes, 0 "
            "predictions.".format(
                _threshold_used,
                "the intended threshold" if error_threshold_info["used_ceiling"]
                else "already the adaptive fallback -- every detection scores lower",
                sum(len(v) for v in gt_by_image.values())))
        panels = []
    if panels:
        figure = figures.overlay_grid(panels, columns=4, panel_height=2.0,
                                      title="Failure gallery. Blue solid: ground truth, "
                                            "orange dashed: prediction")
        figures.save_figure(figure, "failure_gallery",
                            "Representative failures of the full model on the DENTEX "
                            "test split, one row per failure mode.{}".format(
                                _threshold_caption_note),
                            "03_evaluate_and_build_assets", run.mode, "figure:qualitative",
                            inputs=[predictions_path], note=json.dumps(gallery_ids))

    if drawable_predictions:
        with open(os.path.join(paths["subsets"], "clean_stress.json")) as handle:
            subsets = json.load(handle)
        curated = subsets["clean"]["image_ids"][:6] + subsets["stress"]["image_ids"][:6]
        panels = [panel(i, "{} ({})".format(
            by_id[i]["file_name"],
            "clean" if i in subsets["clean"]["image_ids"] else "stress"))
            for i in curated if i in by_id]
        figure = figures.overlay_grid(
            panels, columns=4, panel_height=2.0,
            title="Prediction vs ground truth, 12 curated test images")
        figures.save_figure(figure, "qualitative_overlays",
                            "Full-model predictions (orange, dashed) against ground "
                            "truth (blue, solid) on six clean and six stress-subset "
                            "test images.{}".format(_threshold_caption_note),
                            "03_evaluate_and_build_assets", run.mode,
                            "figure:qualitative",
                            inputs=[predictions_path], note=json.dumps(curated))
    print(json.dumps(gallery_ids, indent=2))
else:
    # These two classes are produced only by the GPU sections above. Without an
    # explicit "not run" record, a CPU-only pass fails the contract check at the
    # very end -- after every table and figure it CAN build has been rebuilt.
    # The contract requires each class to be ACCOUNTED FOR, not present.
    for asset_class in ("figure:error_clusters", "figure:qualitative"):
        figures.record_not_run(
            asset_class, "03_evaluate_and_build_assets", run.mode,
            "requires a GPU session: error analysis and the qualitative "
            "overlays need model inference over the test images")


In [ ]:
# ---- CPU or GPU inference cost, whichever this session can measure ----
CPU_IMAGES = 20
if FULL_WEIGHTS and HAS_GPU:
    for step in (1, 4):
        output = os.path.join(setup_env.RUNS_DIR, "predictions",
                              "full_gpu_step{}.json".format(step))
        measured = eval_utils.dump_predictions(
            FULL_WEIGHTS, CFG["diagnosis"], "diagnosis_test", 2, output, seed=0,
            device="cuda", sample_step=step, limit=run.eval_limit)
        measured.update(eval_utils.model_size_report(FULL_WEIGHTS))
        eval_utils.save_result(
            manifest.result_name("lowresource", device="cuda", step=step), measured)
        print("step {}: {:.3f} s/image, {:.1f} img/min, peak VRAM {} MB, {} M params".format(
            step, measured["mean_seconds_per_image"], measured["images_per_minute"],
            measured.get("peak_gpu_memory_mb"), measured["parameters_millions"]))
elif FULL_WEIGHTS:
    output = os.path.join(setup_env.RUNS_DIR, "predictions", "full_cpu20.json")
    measured = eval_utils.dump_predictions(
        FULL_WEIGHTS, CFG["diagnosis"], "diagnosis_test", 2, output, seed=0,
        device="cpu", limit=CPU_IMAGES, sample_step=1)
    measured.update(eval_utils.model_size_report(FULL_WEIGHTS))
    eval_utils.save_result(manifest.result_name("lowresource", device="cpu"), measured)
    print("CPU: {:.2f} s/image over {} images".format(
        measured["mean_seconds_per_image"], measured["images"]))


In [ ]:
# ---- CPU vs GPU: same weights, same seed — do the detections agree? ----
agreement = None
cpu_predictions = os.path.join(setup_env.RUNS_DIR, "predictions", "full_cpu20.json")
gpu_predictions = os.path.join(setup_env.RUNS_DIR, "predictions", "full_gpu_step1.json")
if os.path.exists(cpu_predictions) and os.path.exists(gpu_predictions):
    with open(cpu_predictions) as handle:
        cpu_records = json.load(handle)
    with open(gpu_predictions) as handle:
        gpu_records = json.load(handle)
    shared = {r["image_id"] for r in cpu_records} & {r["image_id"] for r in gpu_records}
    matched_count = total = 0
    for image_id in shared:
        left = [r for r in cpu_records if r["image_id"] == image_id and r["score"] >= 0.5]
        right = [r for r in gpu_records if r["image_id"] == image_id and r["score"] >= 0.5]
        for record in left:
            total += 1
            if any(eval_utils._iou(record["bbox"], other["bbox"]) >= 0.5
                   and record.get("category_id_3") == other.get("category_id_3")
                   for other in right):
                matched_count += 1
    agreement = {"images_compared": len(shared), "cpu_detections": total,
                 "matched_on_gpu": matched_count,
                 "agreement_rate": round(matched_count / total, 4) if total else None,
                 "note": "residual disagreement is CPU/GPU kernel nondeterminism plus "
                         "the stochastic noisy-box start"}
    print(json.dumps(agreement, indent=2))
else:
    print("run this notebook once on GPU and once on CPU to get the agreement check")


In [ ]:
# ---- Load every raw result ----
def collect(kind, key):
    out = {}
    for name in manifest.list_results(kind):
        parts = manifest.parse_result_name(name)
        if parts["kind"] != kind:
            continue
        out[parts.get(key, name)] = eval_utils.load_result(name)
    return out

main_results = collect("eval_main", "model")
step_results = collect("steps", "step")
step_latency = collect("steps_latency", "step")
degradation_results = collect("degradation", "condition")
fault_results = collect("fault", "condition")
runtime_results = collect("runtime", "model")
lowresource = collect("lowresource", "device")
trajectories = collect("trajectory", "variant")
error_results = collect("errors", "model")
subset_results = {}
for name in manifest.list_results("subset"):
    parts = manifest.parse_result_name(name)
    subset_results["{}|{}".format(parts.get("model"), parts.get("subset"))] = \
        eval_utils.load_result(name)
print({k: len(v) for k, v in {
    "main": main_results, "steps": step_results, "degradation": degradation_results,
    "fault": fault_results, "subsets": subset_results, "runtime": runtime_results,
    "trajectory": trajectories, "lowresource": lowresource}.items()})


In [ ]:
# ---- Tables ----
NB = "03_evaluate_and_build_assets"
written = {}

if main_results:
    written["main_results"] = tables.write_table(
        "main_results", tables.main_results_rows(main_results),
        ["tier", "method"] + list(tables.METRIC_COLUMNS) + ["seeds"],
        "Per-tier detection metrics on the DENTEX test split, mean +/- std over "
        "{} inference seeds (RUN_MODE={}).".format(len(run.eval_seeds), run.mode),
        NB, run.mode, "table:main_results")
    written["original_vs_reproduced"] = tables.write_table(
        "original_vs_reproduced", tables.comparison_rows(main_results),
        ["tier", "method", "metric", "original", "reproduced", "reproduced_std",
         "delta", "delta_pct"],
        "Reproduced values against the originals reported in HierarchicalDet "
        "(MICCAI 2023, Table 1). 'not run' marks rows this run mode did not train.",
        NB, run.mode, "table:original_vs_reproduced",
        note="original column is quoted from the paper, not produced here")
    written["seed_variance"] = tables.write_table(
        "seed_variance", tables.seed_variance_rows(main_results),
        ["method", "tier", "metric", "n_seeds", "mean", "std", "min", "max", "values"],
        "Spread across inference seeds. DiffusionDet denoises from random boxes, so "
        "this is inherent inference variance at fixed weights.",
        NB, run.mode, "table:seed_variance")
    written["per_class_ap"] = tables.write_table(
        "per_class_ap", tables.per_class_rows(main_results),
        ["method", "tier", "class", "AP_mean", "n_seeds", "note"],
        "Per-class AP. OUR EXTENSION: the original paper reports tier-level "
        "aggregates only, so there is no reference column.",
        NB, run.mode, "table:main_results")
    # Does the diagnosis-tier manipulation-ablation margin actually come from
    # every class moving together, or from one class carrying it while another
    # moves the opposite way? Exact, not eyeballed off per_class_ap or a chart.
    if "Ours_full" in main_results and "Ours_wo_Manipulation" in main_results:
        margin_rows = tables.manipulation_margin_rows(main_results)
        written["manipulation_margin"] = tables.write_table(
            "manipulation_margin", margin_rows,
            ["class", "Ours_full_AP", "Ours_wo_Manipulation_AP", "delta",
             "share_of_tier_margin_pct", "direction"],
            "Per-class decomposition of the diagnosis-tier noisy-box-manipulation "
            "margin (Ours_full minus Ours_wo_Manipulation). A negative delta means "
            "that class scores HIGHER under the ablation -- the tier-level margin "
            "does not hold class-by-class.",
            NB, run.mode, "table:manipulation_margin")
    else:
        tables.record_not_run(
            "table:manipulation_margin", NB, run.mode,
            "Ours_full and/or Ours_wo_Manipulation not both present in this run mode")
else:
    for asset_class in ("table:main_results", "table:original_vs_reproduced",
                        "table:seed_variance", "table:manipulation_margin"):
        tables.record_not_run(asset_class, NB, run.mode, "no evaluation results found")

# ---- table:dataset_audit — rebuild from notebook 01's retained summary ----
# Notebook 01 is a separate kernel from this one and only ever builds this
# table itself, in-session, from its own `audits` dict. If that dict never
# reaches this session (notebook 01 wasn't rerun alongside this one, or its
# `paper_assets/tables/dataset_audit.*` output didn't carry over), the table
# silently disappears even though the data behind it -- the per-tier class
# histograms that back the Deep Caries test-split finding -- is sitting right
# there in `results_raw/<mode>/summary_01_setup_and_data.json`. Rebuild from
# that retained summary the same way the Runs table falls back to retained
# run records, instead of only ever trusting the in-session `audits` dict.
# Static and data-free, so it is always producible regardless of session.
written["label_scheme"] = tables.write_table(
    "label_scheme", tables.label_scheme_rows(),
    ["code", "turkish_word", "gloss", "task_class", "note"],
    "The DENTEX test split's 9-code Turkish clinical labelling scheme. "
    "task_class marks the 3 codes (1/6/7) the diagnosis task actually uses; "
    "Deep Caries has no code in this scheme at all.",
    NB, run.mode, "table:label_scheme")

data_summary_01 = setup_env.read_notebook_summary("01_setup_and_data") or {}
audits_from_summary = data_summary_01.get("audits")
if audits_from_summary:
    written["dataset_audit"] = tables.write_table(
        "dataset_audit", tables.audit_rows(audits_from_summary),
        ["split", "images", "annotations", "images_without_annotations",
         "quadrant_labels", "enumeration_labels", "diagnosis_labels",
         "distinct_resolutions", "unreadable_images", "missing_image_files",
         "malformed_boxes"],
        "DENTEX composition and integrity, as received and converted by this study.",
        NB, run.mode, "table:dataset_audit")
    _histogram_rows = tables.diagnosis_label_histogram_rows(audits_from_summary)
    _histogram_columns = ["split"] + sorted({
        key for row in _histogram_rows for key in row if key != "split"
    })
    written["diagnosis_label_histogram"] = tables.write_table(
        "diagnosis_label_histogram", _histogram_rows, _histogram_columns,
        "Diagnosis-tier class counts per split, straight from the converted "
        "annotations. A class missing from a split's columns had zero "
        "occurrences in that split.",
        NB, run.mode, "table:diagnosis_label_histogram")
    _overlap_from_summary = data_summary_01.get("image_overlap")
    if _overlap_from_summary:
        written["image_overlap"] = tables.write_table(
            "image_overlap",
            [{"metric": key, "value": value} for key, value in _overlap_from_summary.items()
             if key != "per_split_unique_files"]
            + [{"metric": "unique_files__{}".format(name), "value": count}
               for name, count in _overlap_from_summary.get("per_split_unique_files", {}).items()],
            ["metric", "value"],
            "Cross-tier / cross-split image duplication by exact content hash (MD5). "
            "*_overlap_with_training counts diagnosis-tier test/val images that are "
            "byte-identical to an image somewhere in the pooled training tiers.",
            NB, run.mode, "table:image_overlap")
    else:
        tables.record_not_run("table:image_overlap", NB, run.mode,
                              "notebook 01's summary has no image_overlap field -- "
                              "rerun notebook 01 to produce it")
    _geometry_from_summary = data_summary_01.get("quadrant_box_geometry")
    if _geometry_from_summary:
        written["quadrant_box_geometry"] = tables.write_table(
            "quadrant_box_geometry", tables.quadrant_geometry_rows(_geometry_from_summary),
            ["quadrant_category_id", "n_boxes", "mean_area_fraction_of_image",
             "mean_center_x", "mean_center_y"],
            "Tier-0 quadrant box geometry (train split). "
            "mean_area_fraction_of_image near 0.25 would mean a literal "
            "quarter-image region; well below that means a tight bounding box "
            "around the visible teeth in that quadrant.",
            NB, run.mode, "table:quadrant_box_geometry")
    else:
        tables.record_not_run("table:quadrant_box_geometry", NB, run.mode,
                              "notebook 01's summary has no quadrant_box_geometry field -- "
                              "rerun notebook 01 to produce it")
else:
    for asset_class in ("table:dataset_audit", "table:diagnosis_label_histogram",
                        "table:image_overlap", "table:quadrant_box_geometry"):
        tables.record_not_run(
            asset_class, NB, run.mode,
            "results_raw/{}/summary_01_setup_and_data.json is not present in "
            "this session -- rerun notebook 01 (CPU only, quota-free) at least "
            "once alongside this notebook so its summary carries the retained "
            "audit into this build".format(run.mode))

for name, payloads, axis, asset_class, caption in (
    ("step_sweep", step_results, "sampling_steps", "table:step_sweep",
     "Accuracy against diffusion sampling steps, full model."),
    ("degradation", degradation_results, "condition", "table:degradation",
     "Accuracy under image degradation, full model, test split."),
    ("clean_vs_stress", subset_results, "model_subset", "table:clean_vs_stress",
     "Clean versus stress evaluation subsets (selection rule in the dataset audit)."),
    ("fault_injection", fault_results, "condition", "table:fault_injection",
     "Diagnosis-tier accuracy when the prior-tier boxes feeding noisy-box "
     "manipulation are jittered or dropped."),
):
    if not payloads:
        tables.record_not_run(asset_class, NB, run.mode, "experiment not run in this mode")
        continue
    extra = None
    columns = [axis, "tier"] + list(tables.METRIC_COLUMNS)
    if name == "step_sweep" and step_latency:
        extra = {k: {"seconds_per_image":
                     (step_latency.get(k) or {}).get("mean_seconds_per_image")}
                 for k in payloads}
        columns.append("seconds_per_image")
    written[name] = tables.write_table(name, tables.sweep_rows(payloads, axis, extra),
                                       columns, caption, NB, run.mode, asset_class)

# Is a degradation condition beating the clean baseline a reproducible effect
# or averaging-hidden noise? Pair each condition against clean by inference
# seed (same weights, same seed, only the input image differs) instead of
# comparing the two conditions' means.
if degradation_results and "clean" in degradation_results:
    written["degradation_seed_consistency"] = tables.write_table(
        "degradation_seed_consistency",
        tables.degradation_seed_consistency_rows(degradation_results),
        ["condition", "n_seeds_paired", "mean_delta_vs_clean", "min_delta",
         "max_delta", "consistent_direction"],
        "Diagnosis-tier AP, each degraded condition minus clean, paired by "
        "inference seed. consistent_direction=True means every paired seed "
        "agreed in sign -- a property noise does not have.",
        NB, run.mode, "table:degradation_seed_consistency")
else:
    tables.record_not_run("table:degradation_seed_consistency", NB, run.mode,
                          "no degradation results, or no clean baseline among them")

if runtime_results:
    written["failure_counts"] = tables.write_table(
        "failure_counts", tables.failure_rows(runtime_results),
        ["method", "images", "detections", "images_with_no_detections", "degenerate_box",
         "box_out_of_image", "box_covers_whole_image", "extreme_aspect_ratio", "crashes",
         "median_score", "frac_score_above_0.5"],
        "Inference failure accounting on the test split. median_score and "
        "frac_score_above_0.5 are computed from the same forward passes, at no "
        "extra inference cost -- they make 'no detection clears 0.50 "
        "confidence' a number read off this table instead of an impression "
        "from a column of AP zeros.",
        NB, run.mode, "table:failure_counts")
else:
    tables.record_not_run("table:failure_counts", NB, run.mode, "no runtime results")

low_rows = [{"method": "Ours_full", **payload} for payload in lowresource.values() if payload]
if low_rows:
    written["low_resource"] = tables.write_table(
        "low_resource", tables.low_resource_rows(low_rows),
        ["method", "device", "sample_step", "seconds_per_image", "images_per_minute",
         "peak_gpu_memory_mb", "parameters_millions", "checkpoint_mb", "images"],
        "Low-resource benchmark: CPU and GPU inference cost of the full model.",
        NB, run.mode, "table:low_resource")
else:
    tables.record_not_run("table:low_resource", NB, run.mode, "no benchmark measured yet")


In [ ]:
# ---- Figures ----
if step_results:
    series = []
    ordered_steps = sorted(step_results, key=lambda s: int(s))
    for tier in tables.TIER_ORDER:
        points = [(float(s), (step_results[s]["aggregate"].get(tier, {}).get("AP", {}) or {}))
                  for s in ordered_steps]
        points = [(x, c["mean"], c.get("std") or 0.0) for x, c in points
                  if c.get("mean") is not None]
        if points:
            series.append({"label": tier, "x": [p[0] for p in points],
                           "y": [p[1] for p in points], "yerr": [p[2] for p in points]})
    latency = {int(k): (v or {}).get("mean_seconds_per_image")
               for k, v in step_latency.items() if v}
    figure = figures.ap_vs_steps_figure(series, latency or None)
    figures.save_figure(figure, "ap_vs_steps",
                        "AP against diffusion sampling steps for the full model, with "
                        "measured per-image latency on the right axis. Error bars are "
                        "the std over inference seeds.",
                        NB, run.mode, "figure:ap_vs_steps")
else:
    figures.record_not_run("figure:ap_vs_steps", NB, run.mode, "step sweep not run")

if degradation_results:
    baseline = ((degradation_results.get("clean") or {}).get("aggregate", {})
                .get("diagnosis", {}).get("AP", {}) or {}).get("mean")
    # One panel per corruption type: their severities are different quantities
    # in different units, and JPEG quality even runs the opposite direction.
    by_kind = {}
    for payload in degradation_results.values():
        condition = payload.get("condition") or {}
        if condition.get("kind") in (None, "none"):
            continue
        cell = payload["aggregate"]["diagnosis"]["AP"]
        if cell.get("mean") is None:
            continue
        by_kind.setdefault(condition["kind"], []).append(
            {"severity": float(condition["severity"]), "mean": cell["mean"],
             "std": cell.get("std") or 0.0})
    figure = figures.degradation_figure(by_kind, baseline, "diagnosis AP [0.5:0.95]")
    figures.save_figure(figure, "degradation_curves",
                        "Diagnosis-tier AP under Gaussian blur, JPEG recompression and "
                        "downscaling, one panel per corruption because the severities "
                        "are different quantities in different units. Dotted line is "
                        "the clean baseline (AP={}).".format(baseline),
                        NB, run.mode, "figure:degradation")
else:
    figures.record_not_run("figure:degradation", NB, run.mode, "degradation grid not run")

if error_results:
    # Drawn from the saved error analysis, not from live predictions, so it
    # rebuilds without a GPU like every other figure here.
    analysis = error_results.get("Ours_full") or list(error_results.values())[0]
    # A rate over zero localised boxes is undefined, not zero. Results written
    # before error_analysis started saying so report a flat 0.0 for that case,
    # so recover the denominator: labelled_boxes_by_tier when it is there, and
    # the localised-box total when it is not.
    localised = analysis.get("labelled_boxes_by_tier") or {
        str(t): analysis["totals"].get("localised", 0) for t in range(3)}
    rates = [analysis["error_rate_by_tier"].get(str(t)) if localised.get(str(t)) else None
             for t in range(3)]
    if not any(r is not None for r in rates):
        # Nothing to draw, so draw nothing. The panel would be empty whatever
        # is written on it, and an empty bar chart titled "where errors cluster"
        # invites the reader to conclude that errors cluster nowhere. The reason
        # goes in the manifest instead, where it cannot be skimmed past.
        figures.record_not_run(
            "figure:error_clusters", NB, run.mode,
            "no detection scored above {:.2f}, so no box was correctly localised "
            "and the per-tier label error rate is undefined, not zero "
            "({} ground-truth boxes, {} predictions)".format(
                analysis["score_threshold"], analysis["totals"].get("gt", 0),
                analysis["totals"].get("pred", 0)))
    else:
        figure = figures.grouped_bars(
            ["quadrant", "enumeration", "diagnosis"],
            {"wrong label": rates},
            "wrong-label fraction",
            "Where errors cluster across the hierarchy",
            width=figures.SINGLE_COLUMN, rotate=0)
        figures.save_figure(figure, "error_clusters",
                            "Per-tier classification error rate among correctly "
                            "localised detections (IoU >= 0.5) for the full model on "
                            "the test split. Denominator by tier: {}. Detection score "
                            "threshold: {:.4f}{}.".format(
                                ", ".join("{} {}".format(k, v)
                                          for k, v in sorted(localised.items())),
                                analysis["score_threshold"],
                                "" if analysis["score_threshold"] == 0.5
                                else " (lowered from 0.50 -- no detection scored that "
                                     "high at this checkpoint's training budget)"),
                            NB, run.mode, "figure:error_clusters")
else:
    figures.record_not_run("figure:error_clusters", NB, run.mode, "no error analysis")

if fault_results and len({
        json.dumps((p.get("aggregate", {}).get("diagnosis", {}).get("AP", {}) or {})
                   .get("values"))
        for p in fault_results.values()}) == 1:
    # Every severity returned bit-identical per-seed values, which no stochastic
    # perturbation does. It means injection never happened and each "condition"
    # is really the clean baseline. Plotting it plots a flat line captioned as
    # evidence of robustness.
    figures.record_not_run(
        "figure:fault_injection", NB, run.mode,
        "all {} fault conditions returned identical per-seed AP values, so the "
        "prior-tier perturbation was not applied (the detector injects nothing "
        "when no prior box clears the score threshold). The experiment did not "
        "test what it claims to test.".format(len(fault_results)))
elif fault_results:
    series = []
    for axis in ("jitter", "drop"):
        points = []
        for payload in fault_results.values():
            condition = payload.get("condition") or {}
            if condition.get("axis") != axis:
                continue
            cell = payload["aggregate"]["diagnosis"]["AP"]
            points.append((condition.get(axis, 0.0), cell["mean"], cell.get("std") or 0.0))
        if points:
            points.sort()
            series.append({"label": "prior-tier {}".format(axis),
                           "x": [p[0] for p in points], "y": [p[1] for p in points],
                           "yerr": [p[2] for p in points]})
    figure = figures.sweep_figure(series, "perturbation magnitude",
                                  "diagnosis AP [0.5:0.95]",
                                  "Sensitivity to an imperfect prior tier")
    _fault_thresholds = {round((p.get("injection") or {}).get("score_threshold"), 4)
                         for p in fault_results.values()
                         if (p.get("injection") or {}).get("score_threshold") is not None}
    _threshold_note = (
        " Prior-tier score threshold: {:.4f} (lowered from the intended 0.50 -- "
        "no prior-tier box scored that high at this checkpoint's training budget)."
        .format(next(iter(_fault_thresholds)))
        if _fault_thresholds and next(iter(_fault_thresholds)) != 0.5 else "")
    figures.save_figure(figure, "fault_injection",
                        "Diagnosis-tier AP as the enumeration model's detections are "
                        "jittered (normalized-coordinate sigma) or randomly dropped "
                        "before being used as noisy boxes.{}".format(_threshold_note),
                        NB, run.mode, "figure:fault_injection")
else:
    figures.record_not_run("figure:fault_injection", NB, run.mode, "fault injection not run")


In [ ]:
# ---- Per-class bars and the checkpoint-trajectory figure ----
per_class = [r for r in tables.per_class_rows(main_results)
             if r["tier"] in ("enumeration", "diagnosis")]
if per_class:
    for tier in ("enumeration", "diagnosis"):
        rows = [r for r in per_class if r["tier"] == tier]
        if not rows:
            continue
        classes = sorted({r["class"] for r in rows})
        # NaN, not 0.0, when a method has no row for a class: an absent
        # measurement is not a measured zero, and matplotlib draws no bar for it.
        groups = {method: [next((r["AP_mean"] for r in rows
                                 if r["method"] == method and r["class"] == klass),
                                float("nan"))
                           for klass in classes]
                  for method in sorted({r["method"] for r in rows})}
        # A class with NO test ground truth is not a class the model failed on.
        # The released DENTEX test split has no Deep Caries annotations at all
        # (see the data audit), so it must be marked, not left as a blank gap a
        # reader would misread as "never detected". COCOeval already says so:
        # it scores such a class NaN for every model and every seed. Reading it
        # back out of the results rather than re-deriving it from the annotation
        # file keeps the figure a pure function of results_raw/, so it rebuilds
        # on a machine that has no dataset attached.
        absent = [klass for index, klass in enumerate(classes)
                  if all(values[index] != values[index]       # NaN is not equal to itself
                         for values in groups.values())]
        # Dropped from the plot rather than drawn as an empty slot: an
        # unscoreable class has no bar to show at any height, and a gap in the
        # middle of the axis only invites the reader to guess what it means.
        # The caption carries it instead, so it stays on the record.
        if absent:
            print("  {} tier: no test ground truth for {}, dropped from the "
                  "figure".format(tier, absent))
            keep = [index for index, klass in enumerate(classes) if klass not in absent]
            classes = [classes[index] for index in keep]
            groups = {method: [values[index] for index in keep]
                      for method, values in groups.items()}
        figure = figures.grouped_bars(classes, groups, "AP [0.5:0.95]",
                                      "Per-class AP, {} tier (our extension)".format(tier),
                                      rotate=45 if tier == "diagnosis" else 0)
        figures.save_figure(figure, "per_class_ap_{}".format(tier),
                            "Per-class AP at the {} tier. OUR EXTENSION: the original "
                            "paper reports tier-level aggregates only.{}".format(
                                tier,
                                " Not shown: {}, which the released test split does "
                                "not annotate at all, so no AP can be computed for "
                                "it.".format(", ".join(absent)) if absent else ""),
                            NB, run.mode, "figure:per_class_ap")
else:
    figures.record_not_run("figure:per_class_ap", NB, run.mode, "no evaluation results")

ordering, stable = {}, {}
if trajectories:
    shaped = {}
    for variant, payload in trajectories.items():
        per_tier = {}
        for point in payload.get("points", []):
            aggregate = point.get("aggregate")
            tiers = (point.get("result") or {}).get("tiers", {})
            for tier in tables.TIER_ORDER:
                if aggregate:
                    cell = aggregate.get(tier, {}).get("AP", {})
                    value, spread = cell.get("mean"), cell.get("std")
                else:
                    value, spread = tiers.get(tier, {}).get("metrics", {}).get("AP"), 0.0
                if value is not None:
                    per_tier.setdefault(tier, []).append(
                        {"progress": point["progress"], "AP": value, "AP_std": spread})
        for tier in per_tier:
            per_tier[tier].sort(key=lambda p: p["progress"])
        shaped[setup_env.VARIANT_LABELS.get(variant, variant)] = per_tier
    figure = figures.trajectory_figure(shaped, list(tables.TIER_ORDER))
    figures.save_figure(figure, "checkpoint_trajectory",
                        "Per-tier AP against training progress for each diagnosis "
                        "variant. Stable ordering across checkpoints is evidence the "
                        "ablation conclusion is not an artifact of the shortened "
                        "schedule; unstable ordering is itself a finding.",
                        NB, run.mode, "figure:checkpoint_trajectory")
    for tier in tables.TIER_ORDER:
        by_progress = {}
        for variant, per_tier in shaped.items():
            for point in per_tier.get(tier, []):
                by_progress.setdefault(point["progress"], []).append((variant, point["AP"]))
        ordering[tier] = {str(p): [v for v, _ in sorted(vals, key=lambda kv: -kv[1])]
                          for p, vals in sorted(by_progress.items())}
    stable = {tier: len({tuple(order) for order in per_progress.values()}) == 1
              for tier, per_progress in ordering.items()}
    print(json.dumps({"ordering": ordering, "ordering_stable": stable}, indent=2))
else:
    figures.record_not_run("figure:checkpoint_trajectory", NB, run.mode,
                           "no trajectory checkpoints evaluated")


In [ ]:
# ---- repro_checklist.md ----
environment = manifest.load_manifest().get("environment", {})
data_summary = setup_env.read_notebook_summary("01_setup_and_data") or {}
records = [r for r in (training.get("records") or []) if r]
if not records:
    # Same fallback as the trajectory sweep: the summary lives in paper_assets
    # and is easily separated from the checkpoints, but every run_record.json
    # travels with runs/. Without this the checklist's "Compute actually spent"
    # table renders empty, which would understate the study in the paper.
    records = train_utils.discover_runs()

# Do these records describe the checkpoints the evaluation numbers came from?
# An offline rebuild inherits results_raw/ from an earlier session, so a
# training summary NEWER than the evaluation summary means the records document
# a retraining whose checkpoints were never re-evaluated. Saying nothing would
# attribute the inherited numbers to runs that did not produce them.
stale_eval = None
if not HAS_GPU:
    prior_eval = setup_env.read_notebook_summary(NB) or {}
    eval_written = prior_eval.get("written_at")
    train_written = training.get("written_at")
    if eval_written and train_written and train_written > eval_written:
        stale_eval = (eval_written, train_written)
STALE_EVAL_NOTE = (
    "**Provenance: the run records below document the retraining written at {1}; "
    "every evaluation number in this build was produced earlier (written at {0}) "
    "from checkpoints whose training records were not retained. Records and "
    "numbers describe DIFFERENT checkpoints until notebook 03 is re-run against "
    "the retained runs.**".format(*stale_eval) if stale_eval else "")

lines = ["# Reproducibility checklist", "",
         "Generated from executed-run records only.", "",
         "## Environment", "",
         "- repo commit: `{}`".format(environment.get("repo_commit")),
         "- python: {}".format(environment.get("python")),
         "- platform: {}".format(environment.get("platform")),
         "- GPU: {}".format(json.dumps(environment.get("gpu", {}).get("devices", []))),
         "- CUDA / cuDNN: {} / {}".format(environment.get("gpu", {}).get("cuda"),
                                          environment.get("gpu", {}).get("cudnn")),
         "- run mode: **{}**".format(run.mode),
         "- training seed: {} (the repo's own SEED)".format(setup_env.BASE_SEED),
         "- inference seeds: {}".format(list(run.eval_seeds)),
         "- multi-GPU: {}".format(json.dumps(training.get("multi_gpu", {}))), "",
         "## Converted dataset", ""]
_download_info = data_summary.get("download") or {}
if _download_info.get("resolved_revision"):
    lines.append("- source: `{}` at commit `{}` (HF dataset cards can change after "
                 "publication; every finding below is pinned to this exact revision, "
                 "not an undated 'as downloaded')".format(
                     _download_info.get("hf_repo_id"), _download_info["resolved_revision"]))
elif _download_info.get("hf_repo_id"):
    lines.append("- source: `{}`, exact revision could not be resolved ({})".format(
        _download_info.get("hf_repo_id"), _download_info.get("resolved_revision_error")))
for name, digest in (data_summary.get("dataset_hashes") or {}).items():
    lines.append("- `{}`: `{}`".format(name, digest))

lines += ["", "## Runs", ""]
if STALE_EVAL_NOTE:
    lines += [STALE_EVAL_NOTE, ""]
lines += ["| run | config hash | iterations | seed | batch | wall (s) | GPUs | stopped on budget |",
          "|---|---|---|---|---|---|---|---|"]
for record in records:
    lines.append("| {} | `{}` | {} | {} | {} | {} | {} | {} |".format(
        record.get("name"), (record.get("config_hash") or "n/a")[:12],
        record.get("max_iter"), record.get("seed"), record.get("ims_per_batch"),
        record.get("wall_seconds"), record.get("num_gpus"),
        record.get("stopped_on_time_budget")))

# How much of the repo's own documented schedule this run mode actually used.
# RUN_MODE=full carries the only iteration counts we have any citation for --
# the paper itself never states one -- so it is the reference point, not a
# claim about what the original authors ran.
_FULL_SCHEDULE = setup_env.RUN_MODES["full"]
_STAGE_BUDGETS = {
    "quadrant": _FULL_SCHEDULE.quadrant.max_iter,
    "enumeration": _FULL_SCHEDULE.enumeration.max_iter,
    "diagnosis": _FULL_SCHEDULE.diagnosis.max_iter,
}
_budget_rows = []
for stage_name, full_iter in _STAGE_BUDGETS.items():
    stage_records = [r for r in records if stage_name in (r.get("name") or "")]
    if not stage_records or not full_iter:
        continue
    this_iter = stage_records[0].get("max_iter")
    if this_iter is None:
        continue
    _budget_rows.append((stage_name, this_iter, full_iter, this_iter / full_iter))
if _budget_rows:
    lines += ["", "## Budget vs. the repo's own schedule", "",
              "`configs_repro/` ships `RUN_MODE=full` iteration counts "
              "(`quadrant`/`enumeration`: {}, `diagnosis`: {}) as the repo's own "
              "training schedule; the paper text does not state one "
              "independently, so this is the only citable reference point for "
              "how far from convergence this run mode stops.".format(
                  _STAGE_BUDGETS["quadrant"], _STAGE_BUDGETS["diagnosis"]), "",
              "| stage | this run's iterations | `full`-mode iterations | fraction |",
              "|---|---|---|---|"]
    for stage_name, this_iter, full_iter, fraction in _budget_rows:
        lines.append("| {} | {} | {} | {:.1%} |".format(
            stage_name, this_iter, full_iter, fraction))

lines += ["", "## Exact commands", ""]
for record in records:
    command = (record.get("launch") or {}).get("command")
    if command:
        lines.append("```\n{}\n```".format(" ".join(str(c) for c in command)))

lines += ["", "## Remaining nondeterminism", "",
          "- mixed-precision (AMP) reduction order during training",
          "- atomics in the torchvision ROIAlign / NMS CUDA kernels",
          "- DataLoader worker interleaving (NUM_WORKERS=2)",
          "- inference starts from random noisy boxes, which is why every reported "
          "number is a mean over {} inference seeds".format(len(run.eval_seeds)), ""]

checklist = os.path.join(setup_env.PAPER_ASSETS, "repro_checklist.md")
with open(checklist, "w") as handle:
    handle.write("\n".join(lines) + "\n")
manifest.record_asset(checklist, "doc:repro_checklist", NB, run.mode)
manifest.record_asset(setup_env.DEVIATIONS_MD, "doc:deviations", NB, run.mode)
print("\n".join(lines[:35]))


In [ ]:
# ---- scope_and_claims.md: the claim-coverage matrix ----
tested_models = set(main_results)

# A model that scores essentially nothing cannot support a claim, however the
# arms happen to be ordered. "Ours_full beats Ours_wo_Manipulation" was being
# reported as "tested" off 4.97 AP against 2.78 AP, both about a tenth of the
# paper's numbers -- an ordering that undertraining reproduces just as easily as
# the mechanism does. Below this floor a claim is downgraded, so no reader has
# to work out the scale for themselves.
DEGENERATE_AP = 10.0
reference = tables.reference_lookup()


def _headline_ap(model):
    cell = ((main_results.get(model) or {}).get("aggregate", {})
            .get("diagnosis", {}).get("AP", {}) or {})
    return cell.get("mean")


def _too_weak_to_test(models):
    """Models present in this run whose AP is too low to carry a conclusion."""
    weak = []
    for model in models:
        value = _headline_ap(model)
        if value is not None and value == value and value < DEGENERATE_AP:
            paper = (reference.get("diagnosis", {}).get(model, {}) or {}).get("AP")
            weak.append("{} AP {:.2f}{}".format(
                model, value, " vs {:.1f} reported".format(paper) if paper else ""))
    return weak


asset_status = {row["asset_class"]: row["status"] for row in manifest.summary_table()}

coverage = []
for claim in tables.PAPER_CLAIMS:
    status, evidence = claim["default_status"], claim["evidence"]
    unproduced = [a for a in claim.get("requires_assets", [])
                  if asset_status.get(a, "not run") != "produced"]
    if unproduced:
        status = "partially supported"
        evidence = "{}; NOT supported by: {} (recorded as not run)".format(
            evidence, ", ".join(unproduced))
    if claim.get("requires_models"):
        missing = [m for m in claim["requires_models"] if m not in tested_models]
        if missing:
            status = "cited, untested"
            evidence = "not trained in RUN_MODE={} (missing: {})".format(
                run.mode, ", ".join(missing))
        elif status == "tested":
            weak = _too_weak_to_test(claim["requires_models"])
            if weak:
                status = "undertrained, not conclusive"
                evidence = ("{}; every arm is far below the reported scale ({}), so "
                            "the ordering cannot be separated from undertraining"
                            .format(evidence, "; ".join(weak)))
    coverage.append({**claim, "status": status, "evidence": evidence})

# Operator-reported wall-clock for the whole study, kept SEPARATE from measured
# hours and labelled as a recollection when set. None means: report only what
# the run records and result files can prove. (A 22.15 h figure lived here
# briefly and was withdrawn as unreliable -- prefer None over a guess.)
OPERATOR_REPORTED_GPU_HOURS = None

# What the artifacts themselves can account for: training wall time from the
# run records (discover_runs also reads the copies the final training notebook
# retains under results_raw/<mode>/run_records/, deduplicated by stage name),
# plus the wall_seconds every evaluation run stores in its own result file.
training_hours = sum(float(r.get("wall_seconds") or 0.0)
                     for r in records if r.get("device") != "cpu") / 3600.0
eval_seconds = 0.0
for _collection in (main_results, step_results, degradation_results, fault_results,
                    subset_results, runtime_results, trajectories, error_results):
    for _payload in (_collection or {}).values():
        for _run in ((_payload or {}).get("runs") or []):
            if isinstance(_run, dict) and _run.get("device") != "cpu":
                eval_seconds += float(_run.get("wall_seconds") or 0.0)
measured_hours = training_hours + eval_seconds / 3600.0

_compute_line = ("Run mode: **{}**. Measured GPU-hours (training records + "
                 "evaluation results): **{:.2f}**.".format(run.mode, measured_hours))
if OPERATOR_REPORTED_GPU_HOURS is not None:
    _compute_line += (" Operator-reported total for the study: **{:.2f}** "
                      "(recollected, not measured; see Compute actually spent)."
                      .format(OPERATOR_REPORTED_GPU_HOURS))
lines = ["# Scope, claims and what a reviewer can re-verify", "",
         _compute_line, "",
         "## Claim coverage", "",
         "| # | Claim in the original paper | Status | Evidence |", "|---|---|---|---|"]
for index, claim in enumerate(coverage, 1):
    lines.append("| {} | {} | {} | {} |".format(
        index, claim["claim"], claim["status"], claim["evidence"]))

lines += ["", "## Compute actually spent", ""]
if OPERATOR_REPORTED_GPU_HOURS is not None:
    lines += ["Two different things, kept apart on purpose.", ""]
lines += ["| source | GPU-hours | provenance |", "|---|---|---|",
          "| training runs (retained run records) | {:.2f} | measured; "
          "`run_record.json` per stage |".format(training_hours),
          "| evaluation runs in `results_raw/` | {:.2f} | measured; every run "
          "records its own `wall_seconds` |".format(eval_seconds / 3600.0),
          "| total measured | {:.2f} | sum of the two rows above |".format(
              measured_hours)]
if OPERATOR_REPORTED_GPU_HOURS is not None:
    lines.append("| whole study, operator-reported | {:.2f} | **recollected, not "
                 "measured** |".format(OPERATOR_REPORTED_GPU_HOURS))
if STALE_EVAL_NOTE:
    lines += ["", STALE_EVAL_NOTE]
if not training_hours:
    lines += ["",
              "The gap is training. Training ran in earlier sessions whose "
              "`runs/*/run_record.json` files were not carried into this repository, "
              "so the training cost cannot be derived from anything here and is not "
              "independently checkable. Notebook 04 retains those records into "
              "`results_raw/<mode>/run_records/`, which closes the gap."]
lines += ["", "| run | wall (s) | GPUs |", "|---|---|---|"]
for record in records:
    lines.append("| {} | {} | {} |".format(
        record.get("name"), record.get("wall_seconds"), record.get("num_gpus")))
if not records:
    lines.append("| no training run records were retained | n/a | n/a |")

lines += ["", "## Re-verifying without retraining", "",
          "Attach `{}` and `{}`, open `notebooks/03_evaluate_and_build_assets.ipynb` "
          "with Accelerator = GPU T4, set `RUN_MODE = \"{}\"`, and Run All."
          .format(CKPT_DATASET_SLUG, DATA_DATASET_SLUG, run.mode), "",
          "That reproduces every number in `tables/main_results.csv` from the released "
          "checkpoints, with no training. Re-running the same notebook on a CPU session "
          "rebuilds every table and figure for free.", "",
          "## Not a clinical claim", "",
          "Every artifact here is a research artifact produced under a constrained "
          "compute budget. None of it is validated for, or intended for, clinical use.",
          ""]
scope = os.path.join(setup_env.PAPER_ASSETS, "scope_and_claims.md")
with open(scope, "w") as handle:
    handle.write("\n".join(lines) + "\n")
manifest.record_asset(scope, "doc:scope_and_claims", NB, run.mode)
print("\n".join(lines[:30]))


In [ ]:
# ---- Contract check: nothing may be cited that is not in the manifest ----
# ... and nothing in the manifest may be missing from disk. An entry claiming
# "produced" for a file that is not here is a citation a reader cannot follow,
# which is worse than an honest "not run". Downgrade those before asserting.
for asset_class, paths in sorted(manifest.missing_assets().items()):
    print("cited but absent, downgrading to 'not run': {} -> {}".format(
        asset_class, ", ".join(paths)))
    recorder = tables if asset_class.startswith("table:") else figures
    recorder.record_not_run(
        asset_class, NB, run.mode,
        "the manifest cited this asset but the file is not in this repository "
        "({}). It was produced by a session whose output was not carried "
        "across.".format(", ".join(paths)))

contract = manifest.assert_asset_classes()
print(json.dumps(contract, indent=2))
print()
for row in manifest.summary_table():
    print("{:34s} {:8s} {}".format(row["asset_class"], row["status"], row["path"]))


In [ ]:
# ---- Notebook summary (the only cross-notebook contract) ----
summary = {
    "run_mode": run.mode,
    "had_gpu": HAS_GPU,
    "eval_seeds": list(run.eval_seeds),
    "models_evaluated": sorted(main_results),
    "aggregate": {k: v.get("aggregate") for k, v in main_results.items()},
    "cpu_gpu_agreement": agreement,
    "tables": written,
    "contract": contract,
    "ablation_ordering": ordering,
    "ablation_ordering_stable": stable,
    "gpu_hours_measured": measured_hours,
    "gpu_hours_training": training_hours,
    "manifest": manifest.MANIFEST_PATH,
}

path = setup_env.write_notebook_summary("03_evaluate_and_build_assets", summary)
print("wrote", path)
print(json.dumps(summary, indent=2, default=str)[:4000])


In [ ]:
# ---- Publish, last ----
# Round-trips paper_assets/ back into the checkpoint dataset so the next
# session (GPU or CPU) starts with every result this one produced.
publish = {"status": "disabled"}
if PUBLISH_KAGGLE_DATASET:
    publish = train_utils.publish_kaggle_dataset(
        CKPT_DATASET_SLUG, [setup_env.RUNS_DIR, setup_env.PAPER_ASSETS],
        "evaluation + paper assets ({} mode)".format(run.mode))
    summary["kaggle_publish"] = {k: v for k, v in publish.items()
                                 if k not in ("stdout", "stderr")}
    setup_env.write_notebook_summary("03_evaluate_and_build_assets", summary)
print(json.dumps({k: v for k, v in publish.items() if k not in ("stdout", "stderr")},
                 indent=2))
print("\n" + "=" * 72)
print("PAPER ASSETS: {}".format(setup_env.PAPER_ASSETS))
print("=" * 72)
for directory, _subdirs, names in sorted(os.walk(setup_env.PAPER_ASSETS)):
    relative = os.path.relpath(directory, setup_env.PAPER_ASSETS)
    for name in sorted(names):
        print("  {:<56} {:>8.1f} KB".format(
            os.path.join(relative, name).lstrip("./"),
            os.path.getsize(os.path.join(directory, name)) / 1024))
print("\nThese sit at the TOP of /kaggle/working, beside runs/ -- not inside")
print("the cloned repo. Download them from the Output tab.")
